# 06_評估訓練結果.py
**電腦視覺與深度學習 › Day20 YOLO11n 物件偵測(預訓練到自己訓練)**  ·  ⚠️ 錯誤示範(有 bug,對照修正版看)

這支程式讀取訓練時記錄的 results.csv,畫出損失、mAP、precision、recall 四張曲線,幫你判斷模型訓練得好不好。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/7/6)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week2/DAY10"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '06_評估訓練結果.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 修正 data.yaml 路徑(檔案裡寫的是講師電腦的 C:/ 路徑)
import re, pathlib
for y in pathlib.Path('datasets').rglob('data.yaml'):
    t = y.read_text(encoding='utf-8')
    y.write_text(re.sub(r'(?m)^path:.*$', 'path: ' + y.parent.resolve().as_posix(), t), encoding='utf-8')
    print('已修正', y)

### 還沒訓練過?
這支要用 05 訓練出來的結果(`runs/train/`)。Colab 每個 notebook 的檔案不共用,可以執行下一格,在這裡先訓練一次(建議開 T4 GPU,20 回合約 5～10 分鐘)。

In [ ]:
#@title (選用)先在這裡跑一次 05 訓練
!printf '\n\n' | python "05_訓練自己的YOLO.py"

### 訓練會留下哪些檔案
註解列出 Ultralytics 訓練後的檔案:每輪指標的 results.csv、混淆矩陣圖、預測對照圖,以及 best.pt 和 last.pt 兩個權重。

In [ ]:
# 06 — 讀訓練結果，畫關鍵 metric

### 匯入與中文字型
用 pandas 讀表格、matplotlib 畫圖,先設好中文字型。

In [ ]:
#
# 訓練完 Ultralytics 會把所有結果存到 runs/train/<你的 run>/：
#   results.csv           每個 epoch 的 loss / mAP / precision / recall
#   confusion_matrix.png  混淆矩陣（哪些類別會被搞混）
#   labels.jpg            訓練集標籤分布
#   val_batch*.jpg        驗證集預測 vs 真實對照
#   weights/best.pt       最佳模型（拿去推論）
#   weights/last.pt       最後一輪的模型
#
# 這支腳本讀 results.csv，畫學習曲線給你看

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei"]
plt.rcParams["axes.unicode_minus"] = False

### 找最新的訓練紀錄
在兩個可能的位置找所有 results.csv,依修改時間排序,拿最新的一個;一個都沒有就提醒你先跑 05。

In [ ]:
BASE = Path(__file__).parent

# 自動找最新的 results.csv（兩個位置都掃）
search_dirs = [BASE / "runs", BASE.parent.parent / "runs"]
candidates = []
for d in search_dirs:
    if d.exists():
        candidates.extend(d.rglob("results.csv"))
candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
if not candidates:
    raise FileNotFoundError(
        "找不到 results.csv，先跑 05_訓練自己的YOLO.py"
    )
results_csv = candidates[0]
RUN_DIR = results_csv.parent
print(f"讀取：{results_csv}")

### 讀表格
讀進表格。Ultralytics 的欄位名稱前面常有空白,用 strip 清掉,再印出所有欄位。

In [ ]:
df = pd.read_csv(results_csv)
# Ultralytics 的欄位名前面通常有空白，清掉
df.columns = [c.strip() for c in df.columns]

### 第一張:損失曲線
開一個 2 乘 2 的畫布。第一張畫訓練和驗證的框位置損失、類別損失,好的樣子是前幾輪快速下降,之後慢慢變平。

In [ ]:
print("可用欄位：", df.columns.tolist())

# ====== 畫 4 張學習曲線：loss / mAP / precision / recall ======
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(f"訓練結果：{RUN_DIR.name}", fontsize=14, fontweight="bold")

epochs = df["epoch"]

# 1) box loss + cls loss + dfl loss
ax = axes[0, 0]
for col, name in [("train/box_loss", "訓練 box"),
                  ("train/cls_loss", "訓練 cls"),
                  ("val/box_loss",   "驗證 box"),
                  ("val/cls_loss",   "驗證 cls")]:
    if col in df.columns:
        ax.plot(epochs, df[col], label=name)
ax.set_title("Loss 曲線")
ax.set_xlabel("epoch"); ax.set_ylabel("loss")

### 第二張:mAP
畫 mAP 在 IoU 0.5 時的值,以及 0.5 到 0.95 的平均。mAP 越高越好,作業要求 mAP 0.5 至少 0.6。

In [ ]:
ax.legend(); ax.grid(alpha=0.3)

# 2) mAP
ax = axes[0, 1]
for col, name in [("metrics/mAP50(B)",    "mAP@0.5"),
                  ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]:
    if col in df.columns:
        ax.plot(epochs, df[col], label=name, marker="o", markersize=3)
ax.set_title("mAP (越高越好)")
ax.set_xlabel("epoch"); ax.set_ylabel("mAP")

### ⚠️ 第三、四張:precision 與 recall
precision 是模型說有的框裡,真的有的比例,低代表假警報多。recall 是真的東西裡被抓到的比例,低代表漏抓多。注意圖的標題把 precision 叫做準確率,但準確率通常指 accuracy,中文比較常說精確率。

In [ ]:
ax.legend(); ax.grid(alpha=0.3)

# 3) precision
ax = axes[1, 0]
if "metrics/precision(B)" in df.columns:
    ax.plot(epochs, df["metrics/precision(B)"], color="#2ecc71", marker="o", markersize=3)
ax.set_title("Precision (準確率：判為正的裡面真的正的比例)")
ax.set_xlabel("epoch"); ax.set_ylabel("precision")
ax.grid(alpha=0.3)

# 4) recall
ax = axes[1, 1]
if "metrics/recall(B)" in df.columns:
    ax.plot(epochs, df["metrics/recall(B)"], color="#e74c3c", marker="o", markersize=3)
ax.set_title("Recall (召回率：所有真的正的裡面被抓到的比例)")
ax.set_xlabel("epoch"); ax.set_ylabel("recall")
ax.grid(alpha=0.3)

### 顯示並印出最終指標
顯示圖表後,印出最後一輪的四個指標。如果結果不好,程式會提醒你:增加輪數、檢查類別名稱、看混淆矩陣補資料,或做資料增強。

In [ ]:
plt.tight_layout()
plt.show()

# ====== 印最終指標 ======
last = df.iloc[-1]
print()
print("=" * 60)
print(f"最終 epoch {int(last['epoch'])} 指標")
print("=" * 60)
for col in ["metrics/mAP50(B)", "metrics/mAP50-95(B)",
            "metrics/precision(B)", "metrics/recall(B)"]:
    if col in df.columns:
        print(f"  {col:30s} {last[col]:.4f}")

print()
print("如果訓練結果不好：")
print("  - 增加 epochs（05 的 EPOCHS 調到 50~100）")
print("  - 檢查 data.yaml 的類別名是否正確")
print("  - 檢查 confusion_matrix.png 看哪兩類搞混，補資料")
print("  - 資料量太少可以做手動 augmentation 或用 Roboflow 內建的 augmentation")
print()
print(f"混淆矩陣圖：{RUN_DIR / 'confusion_matrix.png'}")
print(f"預測對照：{RUN_DIR / 'val_batch0_pred.jpg'}")

## 小結
你學會看訓練曲線判斷好壞。下一支 07 會把訓練好的 best.pt 接上鏡頭。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/7/6) 看逐句拆解、做練習。